# Day 9: Python System Automator

A modular, standalone Python toolkit for local workspace inspection, file organization, and system telemetry logging. Designed for seamless execution in local environments or Google Colab.

## Features
* **Workspace Inspector (`inspector.py`)**: Recursively discovers and catalogs all Git repositories across your project directories.
* **File Organizer (`organizer.py`)**: Automatically sorts loose files into structured categories (code, documents, archives, configs).
* **System Telemetry (`telemetry.py`)**: Generates structured execution and environment diagnostic logs.

##Architecture: System Automation Pipeline
To formalize Day 9 as a robust, professional module in your workshop, let's establish its clean architecture. Instead of loose scripts, we frame it as an integrated System Automation & Telemetry Pipeline where each component handles a distinct operational layer.

###1. Architectural Overview
```
[ C:\projects Workspace ]
       │
       ├──► 1. INSPECTOR (inspector.py)
       │         └── Discovers & catalogs 18+ local git repositories & file sizes
       │
       ├──► 2. ORGANIZER (organizer.py)
       │         └── Automatically sorts loose root files into structured categories (.py, .md, configs)
       │
       └──► 3. TELEMETRY (telemetry.py)
                 └── Captures platform state, environment paths, and execution timestamps into a structured log

##Module 1: inspector.py (Workspace Scanning)


This module scans your directory tree recursively to catalog every Git repository and audit root file sizes.   

In [ ]:
%%writefile inspector.py
"""
Module: Inspector
Description: Recursively inspects workspace directories and locates Git repositories.
"""

from pathlib import Path
from datetime import datetime

def inspect_workspace_deep(target_dir: str = r"/content") -> None:
    root_path = Path(target_dir)
    if not root_path.exists():
        print(f"Error: Directory {target_dir} does not exist.")
        return

    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f"[{timestamp}] Deep Inspecting workspace: {root_path}\n")

    repos = []
    file_counts = {}
    total_size_bytes = 0

    # Locate every Git repository recursively
    for git_path in root_path.rglob(".git"):
        if git_path.is_dir():
            repo_dir = git_path.parent
            repos.append(repo_dir.relative_to(root_path))

    # Tally loose files in root
    for item in root_path.iterdir():
        if item.is_file():
            ext = item.suffix.lower() or "no_extension"
            file_counts[ext] = file_counts.get(ext, 0) + 1
            total_size_bytes += item.stat().st_size

    # Summary Report
    print("=" * 50)
    print("WORKSPACE INSPECTION REPORT")
    print("=" * 50)
    print(f"Total Local Repositories Found: {len(repos)}")
    print("Discovered Repositories:")
    for repo in sorted(repos):
        print(f"  [REPO] {repo}")

    print(f"\nRoot Workspace Size (loose files): {total_size_bytes / (1024*1024):.2f} MB")
    print("Root File Extensions Breakdown:")
    for ext, count in sorted(file_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"  - {ext}: {count} file(s)")
    print("=" * 50)

if __name__ == "__main__":
    inspect_workspace_deep()

##Module 2: organizer.py (Automated File Classification)

This module manages loose files by sorting them into dedicated extension directories (.py, .md, .json, .zip), eliminating manual clutter.

In [ ]:
%%writefile organizer.py
"""
Module: Organizer
Description: Automatically organizes loose files into categorized subfolders.
"""

from pathlib import Path
import shutil
import sys

EXTENSION_MAPPING = {
    "code": [".py", ".js", ".sh", ".ps1", ".ipynb"],
    "documents": [".md", ".txt", ".pdf", ".docx", ".csv"],
    "archives": [".zip", ".tar", ".gz", ".rar"],
    "configs": [".json", ".yaml", ".yml", ".env", ".ini"]
}

def organize_directory(target_dir: str = r"/content/sample_data") -> None:
    root_path = Path(target_dir)
    if not root_path.exists():
        print(f"Error: Directory {target_dir} does not exist.")
        return

    print(f"Organizing loose files in: {root_path}\n")
    moved_count = 0

    # Check if __file__ is defined in the current environment
    current_file = globals().get("__file__")

    for item in root_path.iterdir():
        # Skip folders and avoid moving the script itself if running as a script
        if item.is_file() and (current_file is None or item.resolve() != Path(current_file).resolve()):
            ext = item.suffix.lower()
            category = "misc"

            for cat, exts in EXTENSION_MAPPING.items():
                if ext in exts:
                    category = cat
                    break

            target_folder = root_path / category
            target_folder.mkdir(exist_ok=True)

            destination = target_folder / item.name
            shutil.move(str(item), str(destination))
            print(f"  [Moved] {item.name} -> {category}/")
            moved_count += 1

    print(f"\nSuccessfully organized {moved_count} file(s).")

if __name__ == "__main__":
    organize_directory()

##Module 3: telemetry.py (System Telemetry & Logging)

This module captures system state information and writes structured telemetry logs.

In [ ]:
%%writefile telemetry.py
"""
Module: Telemetry
Description: Captures system path environment and runtime diagnostic logs.
"""

import sys
import os
from pathlib import Path
from datetime import datetime

def log_system_telemetry(log_filename: str = "system_telemetry.log") -> None:
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    log_path = Path(log_filename)

    telemetry_data = [
        f"=== SYSTEM TELEMETRY REPORT ===",
        f"Timestamp: {timestamp}",
        f"Python Version: {sys.version}",
        f"Platform: {sys.platform}",
        f"Current Working Directory: {os.getcwd()}",
        f"Environment PATH entries: {len(os.environ.get('PATH', '').split(os.pathsep))}",
        f"=" * 30 + "\n"
    ]

    report_text = "\n".join(telemetry_data)
    log_path.write_text(report_text, encoding="utf-8")
    print(report_text)
    print(f"Telemetry log successfully written to: {log_path.absolute()}")

if __name__ == "__main__":
    log_system_telemetry()

##Module 4: README.md (Proof of Work Documentation)

##2. Execution Flow (The Pipeline Script)
You can tie these three modules together with a master orchestration script called main.py so that running a single command executes the full inspection, organization, and logging workflow.

Create a file named main.py in your Day 9 repository:

In [ ]:
"""
Master Orchestration Script - Day 09: System Automation Pipeline
Description: Executes workspace inspection, file organization, and telemetry logging sequentially.
"""

from inspector import inspect_workspace_deep
from organizer import organize_directory
from telemetry import log_system_telemetry

def run_pipeline():
    print("==================================================")
    print("STARTING DAY 09 SYSTEM AUTOMATION PIPELINE")
    print("==================================================\n")

    # Step 1: Inspect repository structure and workspace health
    print("[Step 1/3] Running Deep Workspace Inspection...")
    inspect_workspace_deep()

    print("\n" + "-"*50 + "\n")

    # Step 2: Organize loose files into structured categories
    print("[Step 2/3] Running File Organization...")
    organize_directory()

    print("\n" + "-"*50 + "\n")

    # Step 3: Capture system telemetry and write execution logs
    print("[Step 3/3] Generating System Telemetry Logs...")
    log_system_telemetry()

    print("\n==================================================")
    print("PIPELINE EXECUTION COMPLETE")
    print("==================================================")

if __name__ == "__main__":
    run_pipeline()

##The Gradio Interface (app.py)
Create a file named app.py in your repository:

In [ ]:
"""
Gradio Web App - Day 09: System Telemetry & Log Inspector
Description: Provides a web interface to process, analyze, and format system diagnostic logs.
"""

import gradio as gr
from datetime import datetime

def analyze_system_log(raw_log_text: str) -> str:
    """
    Simulates parsing and formatting raw telemetry or workspace logs.
    """
    if not raw_log_text.strip():
        return "Error: Please provide valid log content to analyze."

    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    lines = raw_log_text.strip().split("\n")
    line_count = len(lines)

    analysis = [
        f"=== HUGGING FACE TELEMETRY ANALYSIS ===",
        f"Processed at: {timestamp}",
        f"Total Log Lines Analyzed: {line_count}",
        f"Status: Clean and Parsed Successfully",
        "-" * 40,
        "EXTRACTED CONTENT SUMMARY:",
    ]

    for i, line in enumerate(lines[:10], 1):
        analysis.append(f"  [{i}] {line}")

    if line_count > 10:
        analysis.append(f"  ... and {line_count - 10} more line(s).")

    return "\n".join(analysis)

# Build the Gradio UI
demo = gr.Interface(
    fn=analyze_system_log,
    inputs=gr.Textbox(lines=8, label="Paste System Telemetry or Log Text", placeholder="Paste your log text here..."),
    outputs=gr.Textbox(label="Analysis Report"),
    title="Day 09: System Telemetry Inspector",
    description="A lightweight cloud utility to parse, inspect, and summarize system logs via Gradio."
)

if __name__ == "__main__":
    demo.launch()